<a href="https://colab.research.google.com/github/langchain-samples/lc-colab-workshops/blob/main/notebooks/05_subagents.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 05 · Subagents: delegation and context isolation

Back to the research agent. In lesson 01 you saw its trace: by step 15, every search result it
had ever seen was still being re-sent on every model call.

Subagents fix that — but not for the reason people usually assume. A subagent is **a context
boundary first and a specialist second.**

**New in this lesson:** `SubAgent`, the `task` tool, per-subagent models and tools, and when
delegation is the wrong call.

> **Need a key?** You need a LangSmith API key stored in Colab Secrets (🔑 in the left
> sidebar) as `LANGSMITH_API_KEY`, with **"Notebook access" turned on**. If you have not done
> that yet, run **[00 · Setup](https://colab.research.google.com/github/langchain-samples/lc-colab-workshops/blob/main/notebooks/00_setup.ipynb)** first — it takes 10 minutes and
> checks everything.

In [ ]:
# --- snippet:setup v1 ---
%pip install -qq \
  "deepagents~=0.7.6" \
  "langchain~=1.3.15" \
  "langchain-openai~=1.5.1" \
  "langsmith~=0.11.0"

import os

try:
    from google.colab import userdata

    key = userdata.get("LANGSMITH_API_KEY")
except Exception:  # not on Colab, or secret unavailable
    from getpass import getpass

    key = os.environ.get("LANGSMITH_API_KEY") or getpass("LANGSMITH_API_KEY: ")

os.environ["LANGSMITH_API_KEY"] = key
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "lcw-05-subagents"

# One constant, used everywhere. Models are served by the LangSmith gateway,
# so this key is the only credential the notebook needs.
MODEL = "langsmith:openai/gpt-5.6-luna"
# --- /snippet ---

print("Ready.")

---

## 1. Baseline: one agent doing everything

Run a genuinely broad question through a single agent and measure it.

In [ ]:
import time

from deepagents import create_deep_agent

QUESTION = (
    "Compare LangGraph, Deep Agents, and plain LangChain agents across three dimensions: "
    "control flow, state management, and when each is the right choice. "
    "Research each one before answering."
)

solo = create_deep_agent(
    model=MODEL,
    tools=[{"type": "web_search"}],
    system_prompt="You are a thorough research assistant. Cite sources.",
)

t0 = time.time()
solo_result = solo.invoke({"messages": [{"role": "user", "content": QUESTION}]})
solo_seconds = time.time() - t0

print(f"messages: {len(solo_result['messages'])}")
print(f"seconds:  {solo_seconds:.0f}")
print()
print(solo_result["messages"][-1].text[:600])

In [ ]:
# How much text is the parent carrying around?
from langchain_core.messages.utils import count_tokens_approximately

def history_size(result):
    return count_tokens_approximately(result["messages"])

print(f"solo agent final history: ~{history_size(solo_result):,} tokens")
print()
print("Every one of those tokens was re-sent on the last model call.")

---

## 2. The same task, delegated

A `SubAgent` is a dict: a name, a description of when to use it, and its own system prompt. It
may also override its own tools and model.

The parent gets a `task` tool. When it delegates, the subagent runs its own full agent loop in
a **separate context**, and returns only its final message.

In [ ]:
researcher = {
    "name": "researcher",
    "description": (
        "Researches ONE specific topic in depth using web search and returns a dense summary "
        "with citations. Use this for each topic you need to investigate, one call per topic."
    ),
    "system_prompt": (
        "You research exactly one topic thoroughly using web search.\n"
        "Return a dense, factual summary of at most 300 words with inline citations.\n"
        "Do not pad it with caveats or restate the question."
    ),
    "tools": [{"type": "web_search"}],
}

supervisor = create_deep_agent(
    model=MODEL,
    system_prompt=(
        "You coordinate research. For a question spanning multiple topics, delegate each "
        "topic to the `researcher` subagent, then synthesise their summaries into one answer. "
        "Do not research topics yourself."
    ),
    subagents=[researcher],
)

t0 = time.time()
team_result = supervisor.invoke({"messages": [{"role": "user", "content": QUESTION}]})
team_seconds = time.time() - t0

print(f"messages: {len(team_result['messages'])}")
print(f"seconds:  {team_seconds:.0f}")

In [ ]:
print(f"{'':22} {'solo':>10} {'delegated':>10}")
print(f"{'parent messages':22} {len(solo_result['messages']):>10} {len(team_result['messages']):>10}")
print(f"{'parent history tokens':22} {history_size(solo_result):>10,} {history_size(team_result):>10,}")
print(f"{'wall clock (s)':22} {solo_seconds:>10.0f} {team_seconds:>10.0f}")
print()
print(team_result["messages"][-1].text[:600])

The parent's history is typically much smaller, while the *total* work is the same or greater.
That is the trade: **you spend tokens and latency to buy context headroom in the parent.**

### 🧠 Checkpoint

Open the delegated run in LangSmith (`lcw-05-subagents`). Find a `task` tool call and expand
the child run underneath it.

What did the parent receive back, and what did it never see?

<details><summary>Show answer</summary>

The parent received **one message**: the subagent's final summary, a few hundred tokens.

It never saw the subagent's own work — every `web_search` call, every page of raw results,
every intermediate reasoning step. Those existed only inside the child run and were discarded
when the subagent returned.

That is the whole point. Three searches returning 8,000 tokens each cost the parent 24,000
tokens forever in the solo version. Delegated, they cost the parent one 300-word summary.

Note the corollary, which is a real cost: if the parent later needs a detail the subagent saw
but did not include in its summary, **that detail is gone.** The subagent's return message is
the only channel. This is why a subagent's system prompt should specify what to return, not
just what to do.

</details>

---

## 3. Each subagent is configured independently

Cheap model for bulk gathering, stronger model for judgement. Narrow tools for narrow jobs.

In [ ]:
gatherer = {
    "name": "gatherer",
    "description": "Collects raw facts on a topic. Fast and cheap. Use for straightforward lookups.",
    "system_prompt": "Collect facts on the topic. Return bullet points with citations. No analysis.",
    "tools": [{"type": "web_search"}],
    "model": MODEL,  # swap for a smaller/cheaper model in production
}

analyst = {
    "name": "analyst",
    "description": (
        "Analyses already-gathered facts and forms a judgement. Has NO search tools — "
        "pass it the facts in your request."
    ),
    "system_prompt": (
        "You reason about facts you are given. State a clear conclusion and name the "
        "strongest argument against it. You cannot search; work with what you are given."
    ),
    "tools": [],  # deliberately toolless: it cannot wander off and browse
}

two_tier = create_deep_agent(
    model=MODEL,
    system_prompt=(
        "Use `gatherer` to collect facts, then pass those facts to `analyst` for judgement. "
        "Synthesise the result."
    ),
    subagents=[gatherer, analyst],
)

out = two_tier.invoke({"messages": [{"role": "user", "content":
    "Should a team building a customer support bot use Deep Agents or write a LangGraph graph by hand?"
}]})
print(out["messages"][-1].text[:800])

Giving `analyst` **no tools** is a design choice, not an oversight. A subagent with search
available will search, even when you wanted it to think about what it already has.

---

## 4. The subagent you have had since lesson 01

You never passed `subagents=`, but every agent in this course has had a `task` tool: Deep Agents
adds a **general-purpose subagent** by default. It is why lesson 01's agent could hand off a
chunk of work without you configuring anything.

In [ ]:
plain = create_deep_agent(model=MODEL, system_prompt="You are helpful.")

# Ask it directly — `task` will be in the list, alongside the filesystem tools from lesson 02.
print(plain.invoke({"messages": [{"role": "user", "content":
    "List the names of every tool you have available. Just the names, one per line."
}]})["messages"][-1].text)

### 🧠 Checkpoint

You gave `analyst` an empty tool list. Why would giving it web search — which seems strictly
more useful — make it worse at its job?

<details><summary>Show answer</summary>

Because **a subagent with a tool will use that tool.** Given search, the analyst stops analysing
the facts it was handed and starts gathering new ones — which is the gatherer's job, done again,
more expensively, with a model chosen for judgement rather than bulk retrieval.

You also lose the property that made the split worth doing. The point of two tiers is that the
expensive reasoning step sees a small, curated input. An analyst that searches rebuilds the
large context you were avoiding.

The general principle: **a subagent's tool list is a constraint you are imposing on purpose,
not a set of capabilities you are generously granting.** Removing a tool is a design decision as
real as adding one.

</details>

---

## 5. When *not* to delegate

Delegation is not free. Each `task` call is a whole extra agent loop: more latency, more tokens,
and one more place for the intent to get garbled.

Do not delegate when:

- **The task is short.** A single lookup does not need its own context.
- **The subagent would need the full conversation.** Anything requiring nuance from earlier
  turns is better done inline — you would have to re-explain it all anyway.
- **Latency matters.** A user waiting on a chat reply feels every nested loop.
- **The output is hard to summarise.** If you need everything the subagent saw, isolation is
  working against you.

Delegate when the work is **bulky, separable, and summarisable**. Research is the canonical fit.

### ✍️ Exercise

Add a **critic** subagent that the supervisor must consult before finalising.

Requirements:

- it has no search tools — it critiques what it is given
- its description makes clear it is called **last**, on a draft
- the supervisor's prompt requires one critic pass and a revision before answering

Then check the trace: did the final answer actually change after the critique, or did the
supervisor call the critic and ignore it?

<details><summary>Show a solution</summary>

```python
critic = {
    "name": "critic",
    "description": (
        "Reviews a DRAFT answer for unsupported claims, missing counter-arguments, and "
        "vagueness. Call this last, with the full draft. Has no search tools."
    ),
    "system_prompt": (
        "You review draft research answers. List concrete problems as bullets:\n"
        "  - claims with no citation\n"
        "  - important counter-arguments not addressed\n"
        "  - vague statements that should be specific\n"
        "If the draft is genuinely sound, say so in one line. Do not rewrite it."
    ),
    "tools": [],
}

supervisor = create_deep_agent(
    model=MODEL,
    system_prompt=(
        "You coordinate research.\n"
        "1. Delegate each topic to `researcher`.\n"
        "2. Write a draft answer.\n"
        "3. Send the draft to `critic`.\n"
        "4. Revise using the critique, then reply.\n"
        "Never skip step 3."
    ),
    subagents=[researcher, critic],
)

out = supervisor.invoke({"messages": [{"role": "user", "content": QUESTION}]})
print(out["messages"][-1].text)
```

</details>

---

## 📌 Key takeaways

- A subagent is a **context boundary** first and a specialist second.
- The parent sees only the subagent's final message — everything else stays in the child run.
- That isolation cuts both ways: detail the subagent did not summarise is gone for good.
- Each subagent gets its own prompt, tools, and model — a toolless subagent is often a deliberate design.
- `task` is just a tool, and a general-purpose subagent has been present since lesson 01.
- Delegation costs latency and tokens. Delegate work that is bulky, separable, and summarisable.

---

## ➡️ Next

**[06 · Middleware and human-in-the-loop](https://colab.research.google.com/github/langchain-samples/lc-colab-workshops/blob/main/notebooks/06_middleware_and_hitl.ipynb)**

You have changed behaviour by changing prompts and tools. Next: changing it at the **seams** —
retries, summarization, PII redaction, and stopping the agent for human approval before it
issues a refund.